In [ ]:
import random
from datetime import datetime, timedelta
import mysql.connector
import pandas as pd

## Połączenie

In [ ]:
con = mysql.connector.connect(
    host = "giniewicz.it",
    user = "team07",
    password = "te@mlot",
    database = "team07"
)

if(con):
    print("Połączenie udane")
else:
    print("Połączenie nieudane")
con.close()

Połączenie udane


## Customers

In [ ]:
# Funkcje generujące dane
def losowa_data_urodzenia():
    start = datetime(2150, 1, 1)
    end = datetime(2230, 12, 31)
    delta = end - start
    random_days = random.randint(0, delta.days)
    return (start + timedelta(days=random_days)).date()

def generuj_pesel(data_urodzenia, gender):
    rok = data_urodzenia.year
    miesiac = data_urodzenia.month
    dzien = data_urodzenia.day
    
    if 2100 <= rok <= 2199:
        miesiac += 40
    elif 2200 <= rok <= 2299:
        miesiac += 60
    
    if gender == 'M':
        cyfra_plci = random.choice([1, 3, 5, 7, 9])
    else:
        cyfra_plci = random.choice([0, 2, 4, 6, 8])
    
    losowa_czesc = f"{random.randint(0, 999):03d}{cyfra_plci}"
    pesel_bez_kontrolnej = f"{rok % 100:02d}{miesiac:02d}{dzien:02d}{losowa_czesc}"
    
    wagi = [1, 3, 7, 9, 1, 3, 7, 9, 1, 3]
    suma = sum(int(pesel_bez_kontrolnej[i]) * wagi[i] for i in range(10))
    cyfra_kontrolna = (10 - (suma % 10)) % 10
    
    return f"{pesel_bez_kontrolnej}{cyfra_kontrolna}"

# Wczytanie danych z plików CSV
def wczytaj_dane_imion_nazwisk():
    imiona_men_df = pd.read_csv('imiona_men.csv')
    nazwiska_men_df = pd.read_csv('nazwiska_men.csv')
    imiona_woman_df = pd.read_csv('imiona_woman.csv')
    nazwiska_woman_df = pd.read_csv('nazwiska_woman.csv')

    imiona_men = imiona_men_df['IMIĘ_PIERWSZE'].dropna().unique().tolist()
    nazwiska_men = nazwiska_men_df['Nazwisko aktualne'].dropna().unique().tolist()
    imiona_woman = imiona_woman_df['IMIĘ_PIERWSZE'].dropna().unique().tolist()
    nazwiska_woman = nazwiska_woman_df['Nazwisko aktualne'].dropna().unique().tolist()

    sample_size = min(1000, len(imiona_men), len(nazwiska_men), len(imiona_woman), len(nazwiska_woman))
    
    # Losowanie par dla każdej płci
    sampled_imiona_men = random.sample(imiona_men, sample_size)
    sampled_nazwiska_men = random.sample(nazwiska_men, sample_size)
    sampled_imiona_woman = random.sample(imiona_woman, sample_size)
    sampled_nazwiska_woman = random.sample(nazwiska_woman, sample_size)

    # Przygotowanie słowników z danymi
    dane = {
        'M': list(zip(sampled_imiona_men, sampled_nazwiska_men)),
        'F': list(zip(sampled_imiona_woman, sampled_nazwiska_woman))
    }
    
    return dane

def main():
    # Wczytanie danych imion i nazwisk
    dane_imion_nazwisk = wczytaj_dane_imion_nazwisk()
    
    try:
        con = mysql.connector.connect(
            host="giniewicz.it",
            user="team07",
            password="te@mlot", 
            database="team07"
        )
        cursor = con.cursor()

        print("Rozpoczęcie generowania danych...")
        
        licznik_men = 0
        licznik_woman = 0
        
        for i in range(1, 2001):
            gender = random.choice(['M', 'F'])
            
            # Wybór imienia i nazwiska odpowiedniego do płci
            if gender == 'M':
                first_name, last_name = dane_imion_nazwisk['M'][licznik_men % len(dane_imion_nazwisk['M'])]
                licznik_men += 1
            else:
                first_name, last_name = dane_imion_nazwisk['F'][licznik_woman % len(dane_imion_nazwisk['F'])]
                licznik_woman += 1
            
            date_of_birth = losowa_data_urodzenia()
            pesel = generuj_pesel(date_of_birth, gender)
            
            cursor.execute("""
                INSERT INTO Customers 
                (pesel, first_name, last_name, date_of_birth, gender)
                VALUES (%s, %s, %s, %s, %s)
            """, (pesel, first_name, last_name, date_of_birth, gender))
            
            if i % 100 == 0:
                print(f"Wygenerowano {i}/2000 rekordów...")

        con.commit()
        print("Pomyślnie dodano 2000 rekordów do bazy danych.")
        print(f"Podsumowanie: Mężczyźni: {licznik_men}, Kobiety: {licznik_woman}")

    except mysql.connector.Error as err:
        print(f"Błąd podczas łączenia z bazą danych: {err}")
    finally:
        if 'con' in locals() and con.is_connected():
            cursor.close()
            con.close()
            print("Połączenie z bazą danych zostało zamknięte.")

if __name__ == "__main__":
    main()

Rozpoczęcie generowania danych...
Wygenerowano 100/2000 rekordów...
Wygenerowano 200/2000 rekordów...
Wygenerowano 300/2000 rekordów...
Wygenerowano 400/2000 rekordów...
Wygenerowano 500/2000 rekordów...
Wygenerowano 600/2000 rekordów...
Wygenerowano 700/2000 rekordów...
Wygenerowano 800/2000 rekordów...
Wygenerowano 900/2000 rekordów...
Wygenerowano 1000/2000 rekordów...
Wygenerowano 1100/2000 rekordów...
Wygenerowano 1200/2000 rekordów...
Wygenerowano 1300/2000 rekordów...
Wygenerowano 1400/2000 rekordów...
Wygenerowano 1500/2000 rekordów...
Wygenerowano 1600/2000 rekordów...
Wygenerowano 1700/2000 rekordów...
Wygenerowano 1800/2000 rekordów...
Wygenerowano 1900/2000 rekordów...
Wygenerowano 2000/2000 rekordów...
Pomyślnie dodano 2000 rekordów do bazy danych.
Podsumowanie: Mężczyźni: 995, Kobiety: 1005
Połączenie z bazą danych zostało zamknięte.


In [12]:
con = mysql.connector.connect(
    host="giniewicz.it",
    user="team07",
    password="te@mlot",
    database="team07"
)

cursor = con.cursor()

# Pobieramy dane potrzebne do stworzenia e-maila
cursor.execute("SELECT pesel, first_name, last_name FROM Customers")
klienci = cursor.fetchall()  

# Generujemy emaile
dane_do_emaila = []
for pesel, first, last in klienci:
    email = f"{first.lower()}_{last.lower()}@email.com".replace(" ", "").replace("ł", "l").replace("ń", "n")
    dane_do_emaila.append((email, pesel))
# Aktualizujemy kolumnę email
sql = "UPDATE Customers SET email = %s WHERE pesel = %s"
cursor.executemany(sql, dane_do_emaila)

con.commit()
print("Zaktualizowano emaile.")

cursor.close()
con.close()


Zaktualizowano emaile.


In [13]:
con = mysql.connector.connect(
    host="giniewicz.it",
    user="team07",
    password="te@mlot",
    database="team07"
)

cursor = con.cursor()

def losowe_ubezpieczenie():
    return random.choices([True, False], weights=[95, 5], k=1)[0]
cursor.execute("SELECT pesel FROM Customers")
ids = cursor.fetchall()

# Twórz pary (is_insured, pesel)
dane_ubezpieczenia = [(losowe_ubezpieczenie(), id_[0]) for id_ in ids]
sql = "UPDATE Customers SET is_insured = %s WHERE pesel = %s"
cursor.executemany(sql, dane_ubezpieczenia)

con.commit()
print("Zaktualizowano kolumnę is_insured.")

cursor.close()
con.close()

Zaktualizowano kolumnę is_insured.


In [ ]:
# Wczytanie danych z plików CSV
try:
    imiona_woman_df = pd.read_csv('imiona_woman.csv')
    imiona_men_df = pd.read_csv('imiona_men.csv')
    nazwiska_woman_df = pd.read_csv('nazwiska_woman.csv')
    nazwiska_men_df = pd.read_csv('nazwiska_men.csv')
    
    female_names = imiona_woman_df['IMIĘ_PIERWSZE'].dropna().unique().tolist()
    male_names = imiona_men_df['IMIĘ_PIERWSZE'].dropna().unique().tolist()
    female_surnames = nazwiska_woman_df['Nazwisko aktualne'].dropna().unique().tolist()
    male_surnames = nazwiska_men_df['Nazwisko aktualne'].dropna().unique().tolist()

except FileNotFoundError as e:
    print(f"Błąd: Nie znaleziono pliku {e.filename}")
    exit()
except KeyError as e:
    print(f"Błąd: Brak wymaganej kolumny {e} w pliku CSV")
    exit()

# Definicje relacji i ich wag
relations = ["mother", "father", "sister", "brother", "friend", "partner", "aunt", "uncle"]
relation_weights = [20, 20, 10, 10, 25, 5, 5, 5]  

def losowy_numer():
    """Generuje losowy 9-cyfrowy numer telefonu"""
    return ''.join(str(random.randint(0, 9)) for _ in range(9))

def generuj_kontakt(relacja, nazwisko_klienta):
    """
    Generuje realistyczny kontakt awaryjny w zależności od relacji
    :param relacja: rodzaj relacji (np. 'mother', 'friend')
    :param nazwisko_klienta: nazwisko klienta do dziedziczenia w niektórych przypadkach
    :return: imię i nazwisko kontaktu
    """
    if relacja in ["mother", "sister", "aunt"]:
        imie = random.choice(female_names)
        nazwisko = nazwisko_klienta if relacja in ["mother", "sister"] else random.choice(female_surnames)
    elif relacja in ["father", "brother", "uncle"]:
        imie = random.choice(male_names)
        nazwisko = nazwisko_klienta if relacja in ["father", "brother"] else random.choice(male_surnames)
    else:  # friend, partner
        imie = random.choice(female_names + male_names)
        nazwisko = random.choice(female_surnames + male_surnames)
    
    return f"{imie} {nazwisko}"

try:
    con = mysql.connector.connect(
        host="giniewicz.it",
        user="team07",
        password="te@mlot", 
        database="team07"
    )
    cursor = con.cursor()

    cursor.execute("SELECT pesel, last_name FROM Customers")
    rows = cursor.fetchall()

    if not rows:
        print("Brak klientów w bazie danych. Najpierw dodaj klientów.")
        exit()

    dane_do_wstawienia = []
    for pesel, last_name in rows:
        phone = f"{losowy_numer()}"
        emergency_phone = f"+48{losowy_numer()}"
        relacja = random.choices(relations, weights=relation_weights, k=1)[0]
        kontakt = generuj_kontakt(relacja, last_name)
        dane_do_wstawienia.append((phone, kontakt, emergency_phone, relacja, pesel))

    sql = """
        UPDATE Customers
        SET phone_number = %s,
            emergency_contact_name = %s,
            emergency_contact_phone = %s,
            emergency_contact_relation = %s
        WHERE pesel = %s
    """

    cursor.executemany(sql, dane_do_wstawienia)
    con.commit()
    print(f"Zaktualizowano dane kontaktowe dla {len(dane_do_wstawienia)} klientów.")

except mysql.connector.Error as err:
    print(f"Błąd MySQL: {err}")
finally:
    if 'con' in locals() and con.is_connected():
        cursor.close()
        con.close()
        print("Połączenie z bazą danych zostało zamknięte.")

Zaktualizowano dane kontaktowe dla 2000 klientów.
Połączenie z bazą danych zostało zamknięte.


## Spacecrafts


In [ ]:
con = mysql.connector.connect(
    host="giniewicz.it",
    user="team07",
    password="te@mlot",
    database="team07"
)

cursor = con.cursor()
cursor.execute("DESCRIBE Spacecrafts")
for row in cursor.fetchall():
    print(row[0])  # Pierwszy element to nazwa kolumny
cursor.close()
con.close()

vehicle_id
name
type
capacity
manufacturing_date
status


In [44]:
spacecrafts = [
    ("Parker Solar Probe", "shuttle", 12, "2018-05-20", "active"),
    ("Helios I", "lander", 6, "2019-11-03", "active"),
    ("Helios II", "lander", 14, "2020-03-15", "maintenance"),
    ("Voyager 1", "cruise", 50, "2017-07-12", "retired"),
    ("Voyager 2", "cruise", 60, "2020-10-05", "active"),
    ("Titan Runner", "cruise", 100, "2021-01-30", "active"),
    ("Nova Clipper", "rocket", 8, "2023-04-18", "active"),
    ("Pioneer 10", "rocket", 10, "2015-06-09", "maintenance"),
    ("Pioneer 11", "rocket", 15, "2022-09-27", "active"),
    ("Jupiter Crown", "cruise", 80, "2024-02-01", "active")
]

con = mysql.connector.connect(
    host="giniewicz.it",
    user="team07",
    password="te@mlot",
    database="team07"
)

cursor = con.cursor()

sql = """
    INSERT INTO Spacecrafts
    (name, type, capacity, manufacturing_date, status)
    VALUES (%s, %s, %s, %s, %s)
"""

cursor.executemany(sql, spacecrafts)

con.commit()
print("Dodano 10 pojazdów kosmicznych.")

cursor.close()
con.close()


Dodano 10 pojazdów kosmicznych.


## Employees

In [ ]:
con = mysql.connector.connect(
    host="giniewicz.it",
    user="team07",
    password="te@mlot",
    database="team07"
)

cursor = con.cursor()
cursor.execute("DESCRIBE Employees")
for row in cursor.fetchall():
    print(row[0]) 
cursor.close()
con.close()

first_name
last_name
role
specialization
assigned_vehicle_id
salary
date_of_birth
pesel
gender


In [ ]:
def losowa_data_urodzenia():
    start = datetime(2150, 1, 1)
    end = datetime(2230, 12, 31)
    delta = end - start
    random_days = random.randint(0, delta.days)
    return (start + timedelta(days=random_days)).date()

def generuj_pesel(data_urodzenia, gender):
    rok = data_urodzenia.year
    miesiac = data_urodzenia.month
    dzien = data_urodzenia.day
    
    if 2100 <= rok <= 2199:
        miesiac += 40
    elif 2200 <= rok <= 2299:
        miesiac += 60
    
    if gender == 'M':
        cyfra_plci = random.choice([1, 3, 5, 7, 9])
    else:
        cyfra_plci = random.choice([0, 2, 4, 6, 8])
    
    losowa_czesc = f"{random.randint(0, 999):03d}{cyfra_plci}"
    pesel_bez_kontrolnej = f"{rok % 100:02d}{miesiac:02d}{dzien:02d}{losowa_czesc}"
    
    wagi = [1, 3, 7, 9, 1, 3, 7, 9, 1, 3]
    suma = sum(int(pesel_bez_kontrolnej[i]) * wagi[i] for i in range(10))
    cyfra_kontrolna = (10 - (suma % 10)) % 10
    
    return f"{pesel_bez_kontrolnej}{cyfra_kontrolna}"

def wczytaj_dane_imion_nazwisk():
    imiona_men_df = pd.read_csv('imiona_men.csv')
    nazwiska_men_df = pd.read_csv('nazwiska_men.csv')
    imiona_woman_df = pd.read_csv('imiona_woman.csv')
    nazwiska_woman_df = pd.read_csv('nazwiska_woman.csv')

    imiona_men = imiona_men_df['IMIĘ_PIERWSZE'].dropna().unique().tolist()
    nazwiska_men = nazwiska_men_df['Nazwisko aktualne'].dropna().unique().tolist()
    imiona_woman = imiona_woman_df['IMIĘ_PIERWSZE'].dropna().unique().tolist()
    nazwiska_woman = nazwiska_woman_df['Nazwisko aktualne'].dropna().unique().tolist()

    sample_size = min(40, len(imiona_men), len(nazwiska_men), len(imiona_woman), len(nazwiska_woman))
    
    sampled_imiona_men = random.sample(imiona_men, sample_size)
    sampled_nazwiska_men = random.sample(nazwiska_men, sample_size)
    sampled_imiona_woman = random.sample(imiona_woman, sample_size)
    sampled_nazwiska_woman = random.sample(nazwiska_woman, sample_size)

    dane = {
        'M': list(zip(sampled_imiona_men, sampled_nazwiska_men)),
        'F': list(zip(sampled_imiona_woman, sampled_nazwiska_woman))
    }
    
    return dane

def main():
    dane_imion_nazwisk = wczytaj_dane_imion_nazwisk()
    
    try:
        con = mysql.connector.connect(
            host="giniewicz.it",
            user="team07",
            password="te@mlot", 
            database="team07"
        )
        cursor = con.cursor()

        print("Rozpoczęcie generowania danych...")
        
        licznik_men = 0
        licznik_woman = 0
        
        for i in range(1, 41):
            gender = random.choice(['M', 'F'])
            
            if gender == 'M':
                first_name, last_name = dane_imion_nazwisk['M'][licznik_men % len(dane_imion_nazwisk['M'])]
                licznik_men += 1
            else:
                first_name, last_name = dane_imion_nazwisk['F'][licznik_woman % len(dane_imion_nazwisk['F'])]
                licznik_woman += 1
            
            date_of_birth = losowa_data_urodzenia()
            pesel = generuj_pesel(date_of_birth, gender)
            
            cursor.execute("""
                INSERT INTO Employees 
                (pesel, first_name, last_name, date_of_birth, gender)
                VALUES (%s, %s, %s, %s, %s)
            """, (pesel, first_name, last_name, date_of_birth, gender))
            
            if i % 1 == 0:
                print(f"Wygenerowano {i}/40 rekordów...")

        con.commit()
        print("Pomyślnie dodano 2000 rekordów do bazy danych.")
        print(f"Podsumowanie: Mężczyźni: {licznik_men}, Kobiety: {licznik_woman}")

    except mysql.connector.Error as err:
        print(f"Błąd podczas łączenia z bazą danych: {err}")
    finally:
        if 'con' in locals() and con.is_connected():
            cursor.close()
            con.close()
            print("Połączenie z bazą danych zostało zamknięte.")

if __name__ == "__main__":
    main()

Rozpoczęcie generowania danych...
Wygenerowano 1/2000 rekordów...
Wygenerowano 2/2000 rekordów...
Wygenerowano 3/2000 rekordów...
Wygenerowano 4/2000 rekordów...
Wygenerowano 5/2000 rekordów...
Wygenerowano 6/2000 rekordów...
Wygenerowano 7/2000 rekordów...
Wygenerowano 8/2000 rekordów...
Wygenerowano 9/2000 rekordów...
Wygenerowano 10/2000 rekordów...
Wygenerowano 11/2000 rekordów...
Wygenerowano 12/2000 rekordów...
Wygenerowano 13/2000 rekordów...
Wygenerowano 14/2000 rekordów...
Wygenerowano 15/2000 rekordów...
Wygenerowano 16/2000 rekordów...
Wygenerowano 17/2000 rekordów...
Wygenerowano 18/2000 rekordów...
Wygenerowano 19/2000 rekordów...
Wygenerowano 20/2000 rekordów...
Wygenerowano 21/2000 rekordów...
Wygenerowano 22/2000 rekordów...
Wygenerowano 23/2000 rekordów...
Wygenerowano 24/2000 rekordów...
Wygenerowano 25/2000 rekordów...
Wygenerowano 26/2000 rekordów...
Wygenerowano 27/2000 rekordów...
Wygenerowano 28/2000 rekordów...
Wygenerowano 29/2000 rekordów...
Wygenerowano 30/20

In [ ]:
# Zdefiniowane dane
roles = ["pilot", "engineer", "technician", "commander"]
specializations = ["navigation", "propulsion", "life support", "AI systems", "research"]
salary_ranges = {
    "pilot": (15000, 22000),
    "engineer": (12000, 18000),
    "technician": (8000, 13000),
    "commander": (20000, 30000),
}

# Logika przypisywania specjalizacji
def get_specialization(role):
    if role == "pilot":
        return random.choice(["navigation", "propulsion"])
    elif role == "engineer":
        return random.choice(["propulsion", "life support", "AI systems"])
    elif role == "technician":
        return random.choice(["life support", "AI systems"])
    elif role == "commander":
        return random.choice(["navigation", "research"])
    return None

def update_employees():
    try:
        con = mysql.connector.connect(
            host="giniewicz.it",
            user="team07",
            password="te@mlot", 
            database="team07"
        )
        cursor = con.cursor()

        cursor.execute("SELECT pesel FROM Employees")
        employees = cursor.fetchall()

        for (pesel,) in employees:
            role = random.choice(roles)
            specialization = get_specialization(role)
            
            salary_min, salary_max = salary_ranges[role]
            salary = round(random.uniform(salary_min, salary_max), 2)
            
            cursor.execute("""
                UPDATE Employees
                SET role = %s,
                    specialization = %s,
                    salary = %s
                WHERE pesel = %s
            """, (role, specialization, salary, pesel))

        con.commit()
        print(f"Zaktualizowano {len(employees)} pracowników")

    except Exception as e:
        print(f"Błąd: {e}")
        if 'con' in locals() and con.is_connected():
            con.rollback()
    finally:
        if 'con' in locals() and con.is_connected():
            cursor.close()
            con.close()

update_employees()

Zaktualizowano 40 pracowników


## Destinations

In [ ]:
con = mysql.connector.connect(
    host="giniewicz.it",
    user="team07",
    password="te@mlot",
    database="team07"
)

cursor = con.cursor()
cursor.execute("DESCRIBE Destinations")
for row in cursor.fetchall():
    print(row[0])  # Pierwszy element to nazwa kolumny
cursor.close()
con.close()

destination_id
name
description
range
x_coordinate
y_coordinate
z_coordinate
gravity_level
radiation_risk_level
total_flight_cost


In [ ]:
con = mysql.connector.connect(
    host="giniewicz.it",
    user="team07",
    password="te@mlot",
    database="team07"
)
cursor = con.cursor()

destinations = [
    {
        "name": "Mercury", "description": "Closest planet to the Sun, scorching hot surface.",
        "range": 91.7, "gravity": 0.38, "radiation": 9, "cost": 1500000
    },
    {
        "name": "Venus", "description": "Toxic atmosphere and intense pressure.",
        "range": 41.4, "gravity": 0.9, "radiation": 8, "cost": 1200000
    },
    {
        "name": "Earth Orbit Station", "description": "Low-Earth orbit station for layovers.",
        "range": 0.4, "gravity": 0.0, "radiation": 3, "cost": 100000
    },
    {
        "name": "Moon Base Alpha", "description": "First permanent base on the Moon.",
        "range": 0.384, "gravity": 0.17, "radiation": 5, "cost": 500000
    },
    {
        "name": "Mars", "description": "Red Planet, destination for colonization.",
        "range": 78, "gravity": 0.38, "radiation": 6, "cost": 2000000
    },
    {
        "name": "Phobos Outpost", "description": "Scientific station on Mars' moon Phobos.",
        "range": 78.3, "gravity": 0.01, "radiation": 7, "cost": 800000
    },
    {
        "name": "Ganymede Research Hub", "description": "Frozen moon of Jupiter with subsurface ocean.",
        "range": 628.3, "gravity": 0.15, "radiation": 8, "cost": 3000000
    },
    {
        "name": "Jupiter", "description": "Gas giant - flyby only.",
        "range": 628.7, "gravity": 2.4, "radiation": 10, "cost": 2500000
    },
    {
        "name": "Europa Station", "description": "Landing site on icy moon with potential life.",
        "range": 628.5, "gravity": 0.13, "radiation": 9, "cost": 2800000 
    },
    {
        "name": "Ceres Mining Colony", "description": "Asteroid belt mining and refueling hub.",
        "range": 414, "gravity": 0.03, "radiation": 6, "cost": 1000000
    },
    {
        "name": "Saturn", "description": "Iconic ringed gas giant - orbit cruise only.",
        "range": 1200, "gravity": 1.06, "radiation": 7, "cost": 3500000
    },
    {
        "name": "Titan Base", "description": "Methane lakes and exotic atmosphere.",
        "range": 1220, "gravity": 0.14, "radiation": 6, "cost": 3200000
    },
    {
        "name": "Uranus", "description": "Tilted planet with blue atmosphere.",
        "range": 2600, "gravity": 0.89, "radiation": 6, "cost": 5000000
    },
    {
        "name": "Neptune", "description": "Most distant gas giant with intense storms.",
        "range": 4300, "gravity": 1.14, "radiation": 7, "cost": 6000000
    },
    {
        "name": "Proxima Centauri b Base", "description": "Closest exoplanet to Earth.",
        "range": 401000, "gravity": 1.1, "radiation": 5, "cost": 18000000 
    },
    {
        "name": "The Black Hole Edge Station", "description": "Extreme research post near black hole.",
        "range": 550000, "gravity": 9.8, "radiation": 10, "cost": 50000000
    },
    {
        "name": "Dark Matter Observation Platform", "description": "High-orbit facility for fundamental physics.",
        "range": 200, "gravity": 0.0, "radiation": 2, "cost": 3000000
    },
]

def random_coords():
    return round(random.uniform(-1000, 1000), 2)

destination_data = []
for dest in destinations:
    destination_data.append((
        dest["name"],
        dest["description"],
        dest["range"],
        random_coords(),
        random_coords(),
        random_coords(),
        dest["gravity"],
        dest["radiation"],
        dest["cost"]
    ))

# SQL INSERT
sql = """
INSERT INTO Destinations
(name, description, `range`, x_coordinate, y_coordinate, z_coordinate, gravity_level, radiation_risk_level, total_flight_cost)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)
"""

cursor.executemany(sql, destination_data)
con.commit()
print("Wstawiono 20 destynacji do tabeli Destinations.")

cursor.close()
con.close()


Wstawiono 20 destynacji do tabeli Destinations.



## TripTypes

In [58]:
con = mysql.connector.connect(
    host="giniewicz.it",
    user="team07",
    password="te@mlot",
    database="team07"
)

cursor = con.cursor()
cursor.execute("DESCRIBE TripTypes")
# cursor.execute("SELECT * FROM Destinations")
for row in cursor.fetchall():
    print(row[0])  # Pierwszy element to nazwa kolumny
cursor.close()
con.close()

trip_type_id
name
description
estimated_duration_days
is_one_way


In [ ]:
import mysql.connector

con = mysql.connector.connect(
    host="giniewicz.it",
    user="team07",
    password="te@mlot",
    database="team07"
)
cursor = con.cursor()

# TripTypes data
trip_types = [
    ("Scenic Flight (1 Day)", "A short sightseeing flight over a celestial body.", 1, False),
    ("Short-Term Research Mission", "Scientific expedition with quick return.", 5, False),
    ("Training/Test Flight", "Cadet training and equipment test flight.", 2, False),
    ("Long-Term Research Expedition", "Months-long mission in deep space conditions.", 180, False),
    ("Couples Cosmic Retreat", "Romantic getaway package in space habitat.", 4, False),
    ("Space Emigration", "One-way trip for people relocating off-Earth.", 999, True),
    ("Commercial Shuttle", "Standard interplanetary travel service.", 7, False),
    ("Premium Tourist Expedition", "Extended luxury trip with attractions.", 21, False),
]

# SQL Insert query
sql = """
INSERT INTO TripTypes
(name, description, estimated_duration_days, is_one_way)
VALUES (%s, %s, %s, %s)
"""

# Execute insert
cursor.executemany(sql, trip_types)
con.commit()

print("Inserted 10 trip types into TripTypes table.")

# Clean up
cursor.close()
con.close()


Inserted 10 trip types into TripTypes table.


# Trips

In [2]:
con = mysql.connector.connect(
    host="giniewicz.it",
    user="team07",
    password="te@mlot",
    database="team07"
)

cursor = con.cursor()
cursor.execute("DESCRIBE Trips")
# cursor.execute("SELECT * FROM Destinations")
for row in cursor.fetchall():
    print(row[0])  # Pierwszy element to nazwa kolumny
cursor.close()
con.close()

trip_id
trip_type_id
destination_id
launch_date
return_date
spacecraft_id
launch_method
status
organization_cost


In [6]:
import mysql.connector
import random
from datetime import datetime, timedelta

# Connect to the database
con = mysql.connector.connect(
    host="giniewicz.it",
    user="team07",
    password="te@mlot",
    database="team07"
)
cursor = con.cursor(dictionary=True)

# Reset auto-increment to start from 1
cursor.execute("ALTER TABLE Trips AUTO_INCREMENT = 1")

# Get total_flight_cost per destination_id
cursor.execute("SELECT destination_id, total_flight_cost FROM Destinations")
dest_costs = {row["destination_id"]: row["total_flight_cost"] for row in cursor.fetchall()}

# Get estimated_duration_days and is_one_way per trip_type_id
cursor.execute("SELECT trip_type_id, estimated_duration_days, is_one_way FROM TripTypes")
trip_type_info = {row["trip_type_id"]: (row["estimated_duration_days"], row["is_one_way"]) for row in cursor.fetchall()}

# Options
launch_methods = ["vertical launch", "orbital elevator", "assisted lift"]

# Generate 15 trips
trips = []
for _ in range(15):
    trip_type_id = random.randint(1, 8)
    destination_id = random.choice(list(dest_costs.keys()))
    spacecraft_id = random.randint(1, 10)
    launch_method = random.choice(launch_methods)

    # Launch date between 2023-01-01 and today
    start_date = datetime(2023, 1, 1)
    end_date = datetime.today()
    launch_date = start_date + timedelta(days=random.randint(0, (end_date - start_date).days))
    launch_date_str = launch_date.date().isoformat()

    # Return date and status
    estimated_duration, is_one_way = trip_type_info[trip_type_id]
    today = datetime.today().date()

    if is_one_way:
        return_date_str = None
        status = "ongoing"
    else:
        variation = random.randint(-2, 2)
        return_date = launch_date + timedelta(days=estimated_duration + variation)
        return_date_str = return_date.date().isoformat()
        status = "completed" if return_date.date() < today else "ongoing"

    # Cost
    base_cost = dest_costs[destination_id]
    multiplier = round(random.uniform(0.8, 1.5), 2)
    organization_cost = round(float(base_cost) * multiplier, 2)

    trips.append((
        trip_type_id,
        destination_id,
        launch_date_str,
        return_date_str,
        spacecraft_id,
        launch_method,
        status,
        organization_cost
    ))

# Insert into DB
sql = """
INSERT INTO Trips
(trip_type_id, destination_id, launch_date, return_date, spacecraft_id, launch_method, status, organization_cost)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
"""

cursor.executemany(sql, trips)
con.commit()

print("Inserted 15 trips with correct status and ID reset.")

cursor.close()
con.close()


Inserted 15 trips with correct status and ID reset.


## TripParticipants

In [ ]:
con = mysql.connector.connect(
    host="giniewicz.it",
    user="team07",
    password="te@mlot",
    database="team07"
)

cursor = con.cursor()
cursor.execute("DESCRIBE TripParticipants")
# cursor.execute("SELECT * FROM Destinations")
for row in cursor.fetchall():
    print(row[0])  # Pierwszy element to nazwa kolumny
cursor.close()
con.close()

participant_id
trip_id
customer_id
ticket_price
seat_number


In [9]:
import mysql.connector
import random

# Connect to the database
con = mysql.connector.connect(
    host="giniewicz.it",
    user="team07",
    password="te@mlot",
    database="team07"
)
cursor = con.cursor(dictionary=True)

# Pobierz wszystkie tripy z danymi o pojemności, koszcie i ID
cursor.execute("""
SELECT 
    Trips.trip_id, 
    Trips.destination_id, 
    Trips.spacecraft_id, 
    Destinations.total_flight_cost, 
    Spacecrafts.capacity
FROM Trips
JOIN Destinations ON Trips.destination_id = Destinations.destination_id
JOIN Spacecrafts ON Trips.spacecraft_id = Spacecrafts.vehicle_id
""")
trips_info = cursor.fetchall()

# Pobierz wszystkich klientów
cursor.execute("SELECT customer_id FROM Customers")
customer_ids = [row["customer_id"] for row in cursor.fetchall()]
random.shuffle(customer_ids)

# Generuj wpisy TripParticipants
trip_participants = []
customer_index = 0

for trip in trips_info:
    for seat in range(1, trip["capacity"] + 1):
        # Zapętlenie klientów, jeśli ich zabraknie
        if customer_index >= len(customer_ids):
            customer_index = 0
            random.shuffle(customer_ids)

        customer_id = customer_ids[customer_index]
        customer_index += 1

        # Cena biletu ±15% od kosztu lotu
        price_factor = random.uniform(1, 1.4)
        ticket_price = round(float(trip["total_flight_cost"]) * price_factor/trip["capacity"], 2)

        trip_participants.append((
            trip["trip_id"],
            customer_id,
            ticket_price,
            seat
        ))

# Wstaw dane
sql = """
INSERT INTO TripParticipants (trip_id, customer_id, ticket_price, seat_number)
VALUES (%s, %s, %s, %s)
"""
cursor.executemany(sql, trip_participants)
con.commit()

print(f"✅ Inserted {len(trip_participants)} participants into TripParticipants.")

cursor.close()
con.close()


✅ Inserted 391 participants into TripParticipants.


## Transactions

In [10]:
con = mysql.connector.connect(
    host="giniewicz.it",
    user="team07",
    password="te@mlot",
    database="team07"
)

cursor = con.cursor()
cursor.execute("DESCRIBE Transactions")
for row in cursor.fetchall():
    print(row[0])  # Pierwszy element to nazwa kolumny
cursor.close()
con.close()

transaction_id
customer_id
amount
transaction_date
method
description


In [ ]:
con = mysql.connector.connect(
    host="giniewicz.it",
    user="team07",
    password="te@mlot",
    database="team07"
)
cursor = con.cursor(dictionary=True)

cursor.execute("""
SELECT 
    tp.customer_id,
    tp.ticket_price,
    t.launch_date
FROM TripParticipants tp
JOIN Trips t ON tp.trip_id = t.trip_id
""")
participant_data = cursor.fetchall()

payment_methods = ["credit card", "bank transfer", "crypto", "PayPal"]
descriptions = [
    "Ticket payment",
    "Advance reservation",
    "Final installment",
    "Full fare payment",
    "Booking confirmation"
]

transactions = []
for entry in participant_data:
    customer_id = entry["customer_id"]
    amount = float(entry["ticket_price"])
    launch_date = entry["launch_date"]

    max_days = 365
    min_days = 30
    days_before = random.randint(min_days, max_days)
    transaction_date = launch_date - timedelta(days=days_before)
    transaction_date_str = transaction_date.isoformat()

    method = random.choice(payment_methods)
    description = random.choice(descriptions)

    transactions.append((
        customer_id,
        amount,
        transaction_date_str,
        method,
        description
    ))

sql = """
INSERT INTO Transactions (customer_id, amount, transaction_date, method, description)
VALUES (%s, %s, %s, %s, %s)
"""
cursor.executemany(sql, transactions)
con.commit()

print(f"✅ Inserted {len(transactions)} transactions into Transactions table.")

cursor.close()
con.close()


✅ Inserted 391 transactions into Transactions table.


## VehicleApprovals

In [13]:
con = mysql.connector.connect(
    host="giniewicz.it",
    user="team07",
    password="te@mlot",
    database="team07"
)

cursor = con.cursor()
cursor.execute("DESCRIBE Employees")
for row in cursor.fetchall():
    print(row[0])  # Pierwszy element to nazwa kolumny
cursor.close()
con.close()

employee_id
first_name
last_name
role
specialization
assigned_vehicle_id
salary


In [14]:
import mysql.connector
import pandas as pd
import random

# Wczytywanie imion i nazwisk z plików CSV
imiona_men_df = pd.read_csv('imiona_men.csv')
nazwiska_men_df = pd.read_csv('nazwiska_men.csv')
imiona_woman_df = pd.read_csv('imiona_woman.csv')
nazwiska_woman_df = pd.read_csv('nazwiska_woman.csv')

# Zamiana na listy
imiona_men = imiona_men_df['IMIĘ_PIERWSZE'].dropna().unique().tolist()
nazwiska_men = nazwiska_men_df['Nazwisko aktualne'].dropna().unique().tolist()
imiona_woman = imiona_woman_df['IMIĘ_PIERWSZE'].dropna().unique().tolist()
nazwiska_woman = nazwiska_woman_df['Nazwisko aktualne'].dropna().unique().tolist()

# Losowanie po 2 osoby każdej płci i wymieszanie
sampled_names = list(zip(
    random.sample(imiona_men, 2) + random.sample(imiona_woman, 2),
    random.sample(nazwiska_men, 2) + random.sample(nazwiska_woman, 2)
))
random.shuffle(sampled_names)

# Połączenie z bazą danych
con = mysql.connector.connect(
    host="giniewicz.it",
    user="team07",
    password="te@mlot",
    database="team07"
)
cursor = con.cursor(dictionary=True)

# Pobranie ID pojazdów wg typu
cursor.execute("SELECT vehicle_id, type FROM Spacecrafts")
spacecrafts = cursor.fetchall()

# Grupowanie ID pojazdów wg typu
type_to_ids = {}
for row in spacecrafts:
    type_to_ids.setdefault(row["type"].lower(), []).append(row["vehicle_id"])

# Typy pojazdów i odpowiadające specjalizacje
types_and_specs = {
    "cruise": "Cruise Operations",
    "lander": "Landing Systems",
    "rocket": "Rocket Propulsion",
    "shuttle": "Shuttle Navigation"
}

# Tworzenie pracowników
employees = []
for i, (craft_type, specialization) in enumerate(types_and_specs.items()):
    if craft_type in type_to_ids and type_to_ids[craft_type]:
        assigned_id = random.choice(type_to_ids[craft_type])
        first_name, last_name = sampled_names[i]
        role = "Engineer"
        salary = random.randint(8500, 10500)
        employees.append((first_name, last_name, role, specialization, assigned_id, salary))

# INSERT do tabeli Employees
sql = """
INSERT INTO Employees (first_name, last_name, role, specialization, assigned_vehicle_id, salary)
VALUES (%s, %s, %s, %s, %s, %s)
"""
cursor = con.cursor()
cursor.executemany(sql, employees)
con.commit()

print(f"✅ Inserted {len(employees)} employees assigned to vehicle types.")

cursor.close()
con.close()


✅ Inserted 4 employees assigned to vehicle types.


In [15]:
import mysql.connector
from datetime import datetime, timedelta
import random

# Connect to DB
con = mysql.connector.connect(
    host="giniewicz.it",
    user="team07",
    password="te@mlot",
    database="team07"
)
cursor = con.cursor(dictionary=True)

# Map spacecraft types to employee specializations
type_to_spec = {
    "cruise": "Cruise Operations",
    "lander": "Landing Systems",
    "rocket": "Rocket Propulsion",
    "shuttle": "Shuttle Navigation"
}

approvals = []

for spacecraft_type, specialization in type_to_spec.items():
    # Get the employee assigned to this specialization
    cursor.execute("""
        SELECT employee_id FROM Employees
        WHERE specialization = %s
        LIMIT 1
    """, (specialization,))
    employee = cursor.fetchone()
    if not employee:
        continue

    employee_id = employee["employee_id"]

    # Get all vehicle_ids of this spacecraft type
    cursor.execute("""
        SELECT vehicle_id FROM Spacecrafts
        WHERE type = %s
    """, (spacecraft_type,))
    vehicles = cursor.fetchall()

    for v in vehicles:
        vehicle_id = v["vehicle_id"]
        approval_date = (datetime.today() - timedelta(days=random.randint(5, 60))).date().isoformat()
        approvals.append((vehicle_id, employee_id, approval_date))

# Insert approvals
sql = """
INSERT INTO VehicleApprovals (vehicle_id, employee_id, approval_date)
VALUES (%s, %s, %s)
"""
cursor.executemany(sql, approvals)
con.commit()

print(f"✅ Inserted {len(approvals)} approvals — one employee per type approving all vehicles of that type.")

cursor.close()
con.close()


✅ Inserted 10 approvals — one employee per type approving all vehicles of that type.


In [17]:
approval_notes = [
    "Approved after full inspection",
    "Minor adjustments required",
    "Approved per safety protocol",
    "Conditionally approved",
    "Approved with recommendations"
]


# Connect to DB
con = mysql.connector.connect(
    host="giniewicz.it",
    user="team07",
    password="te@mlot",
    database="team07"
)
cursor = con.cursor(dictionary=True)

# Notes for approval
approval_notes = [
    "Approved after full inspection",
    "Minor adjustments required",
    "Approved per safety protocol",
    "Conditionally approved",
    "Approved with recommendations"
]

# Map spacecraft types to employee specializations
type_to_spec = {
    "cruise": "Cruise Operations",
    "lander": "Landing Systems",
    "rocket": "Rocket Propulsion",
    "shuttle": "Shuttle Navigation"
}

approvals = []

for spacecraft_type, specialization in type_to_spec.items():
    # Get the employee assigned to this specialization
    cursor.execute("""
        SELECT employee_id FROM Employees
        WHERE specialization = %s
        LIMIT 1
    """, (specialization,))
    employee = cursor.fetchone()
    if not employee:
        continue

    employee_id = employee["employee_id"]

    # Get all vehicle_ids of this spacecraft type
    cursor.execute("""
        SELECT vehicle_id FROM Spacecrafts
        WHERE type = %s
    """, (spacecraft_type,))
    vehicles = cursor.fetchall()

    for v in vehicles:
        vehicle_id = v["vehicle_id"]
        approval_date = (datetime.today() - timedelta(days=random.randint(5, 60))).date().isoformat()
        note = random.choice(approval_notes)
        approvals.append((vehicle_id, employee_id, approval_date, note))

# Insert approvals
sql = """
INSERT INTO VehicleApprovals (vehicle_id, employee_id, approval_date, note)
VALUES (%s, %s, %s, %s)
"""
cursor.executemany(sql, approvals)
con.commit()

print(f"✅ Inserted {len(approvals)} approvals with notes.")

cursor.close()
con.close()


✅ Inserted 10 approvals with notes.


In [ ]:
# tylko nietrywialne, bez zbędnych, nie ma sensu zależności przechodnich